# Horizon experiment: diagnostics and figures

How does forecast quality change across horizons of 1, 5, 10, 15 and 20 weekdays,
and why? No inference is rerun and nothing is recomputed here.

**Source of truth.** Every value is computed by `horizon_diagnostics.py` and read
from `results/`. This notebook only reads those CSVs and draws figures.

Only h=20 is the official M6 horizon; 1, 5, 10 and 15 days are extensions outside
the competition specification.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
RESULTS = HERE / "results"
FIGURES = HERE / "visualisations"
FIGURES.mkdir(exist_ok=True)

HORIZONS = [1, 5, 10, 15, 20]
NAIVE_RPS = 0.16
RANK_COLUMNS = [f"Rank{i}" for i in range(1, 6)]
DEGENERATE = (4, 1)   # (round, horizon): 2022-05-30 is US Memorial Day

COLOURS = {"Chronos-2 120M (independent sampling)": "#1f77b4",
           "Financial Chronos 46M": "#2ca02c",
           "Chronos-T5 Base 200M": "#d62728"}
SHORT = {"Chronos-2 120M (independent sampling)": "Chronos-2",
         "Financial Chronos 46M": "Financial Chronos",
         "Chronos-T5 Base 200M": "Chronos-T5 Base"}

summary = pd.read_csv(RESULTS / "horizon_rps_summary.csv")
detail = pd.read_csv(RESULTS / "horizon_forecast_detail.csv")
diagnostics = pd.read_csv(RESULTS / "horizon_diagnostics.csv")
ECE_COLUMN = next(c for c in diagnostics.columns if c.startswith("ece_"))

# the degenerate Memorial Day cell is excluded from every diagnostic
clean = detail[~((detail["round"] == DEGENERATE[0])
                 & (detail["horizon_days"] == DEGENERATE[1]))]

print(f"{len(detail)} detail rows, horizons {sorted(detail['horizon_days'].unique())}")

## Summary metrics

All the numbers, including those no longer plotted. Accuracy is 5-class argmax
accuracy (chance 0.2); entropy is in nats (uniform = ln 5 = 1.609); ECE is the
top-label version with 5 uniform bins; rho is the cross-sectional Spearman
correlation between median predicted and realised returns.

The h=1 row uses the 11-round result that excludes the degenerate Memorial Day
round (see the RPS figure below).

In [ ]:
table = diagnostics.assign(model=diagnostics["model"].map(SHORT)).rename(columns={
    "horizon_days": "horizon",
    "overall_mean_rps_excl_degenerate": "RPS",
    "mean_confidence": "confidence",
    "mean_entropy": "entropy",
    ECE_COLUMN: "ECE",
    "mean_spearman_rho": "rho",
})[["model", "horizon", "RPS", "confidence", "entropy", "accuracy", "ECE", "rho"]]
table = table.sort_values(["model", "horizon"]).round(4).reset_index(drop=True)
table

### White-background image version

The same table rendered as a PNG on a white background, so it can be dropped
straight into a document later.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.2))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.axis("off")

rendered = ax.table(cellText=[[f"{v:.4f}" if isinstance(v, float) else str(v)
                               for v in row] for row in table.to_numpy()],
                    colLabels=table.columns, cellLoc="center", loc="center")
rendered.auto_set_font_size(False)
rendered.set_fontsize(9)
rendered.scale(1, 1.45)

for (row, column), cell in rendered.get_celld().items():
    cell.set_edgecolor("#cccccc")
    if row == 0:
        cell.set_facecolor("#ececec")
        cell.set_text_props(weight="bold")
    elif row % 5 == 0:            # light banding between the three models
        cell.set_facecolor("#f7f7f7")
    else:
        cell.set_facecolor("white")

ax.set_title("Horizon experiment: summary metrics (h=20 is the official M6 horizon)",
             fontsize=11, pad=18)
fig.tight_layout()
fig.savefig(FIGURES / "summary_metrics_table.png", dpi=200, facecolor="white")
plt.show()

## 1. RPS vs horizon

The main performance figure. The ring marks the official 20-day M6 horizon.

At h=1 the small hollow marker is the all-12-round mean and the filled marker is
the 11-round mean that excludes round 4, whose first forecast day (2022-05-30)
was US Memorial Day - about 90 of the 100 assets tie at exactly zero, which
flattens the target and drops even the naive benchmark to 0.024 for that one
cell. The filled marker is the figure's actual series; the hollow one just shows
how large the artefact is.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for model, colour in COLOURS.items():
    block = summary[summary["model"] == model].sort_values("horizon_days")
    y = block["overall_mean_rps_excl_degenerate"]
    ax.plot(block["horizon_days"], y, marker="o", color=colour, label=SHORT[model])
    raw1 = block.loc[block["horizon_days"] == 1, "overall_mean_rps"].iloc[0]
    ax.plot([1], [raw1], marker="o", markersize=4, markerfacecolor="white", color=colour)
    ax.plot([1, 1], [raw1, y.iloc[0]], color=colour, linestyle=":", linewidth=1)
    ax.plot([20], [block.loc[block["horizon_days"] == 20,
                             "overall_mean_rps_excl_degenerate"].iloc[0]],
            marker="o", markersize=13, markerfacecolor="none", color=colour)

ax.axhline(NAIVE_RPS, color="grey", linestyle="--", linewidth=1)
ax.text(20, NAIVE_RPS + 0.0015, "naive 0.160000", color="grey", ha="right", fontsize=9)
ax.set_xscale("log")
ax.set_xticks(HORIZONS)
ax.set_xticklabels([str(h) for h in HORIZONS])
ax.set_xlabel("forecast horizon (weekdays)   -   large ring = official M6 horizon")
ax.set_ylabel("mean RPS across rounds")
ax.set_title("RPS worsens with horizon for all three models")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "rps_vs_horizon.png", dpi=200)
plt.show()

Every model is worse at 20 days than at 1 day, and the model ordering is the same
at every horizon. Chronos-T5 Base degrades most (+0.039 from the corrected h=1 to
h=20) against +0.008 for Chronos-2 and +0.009 for Financial Chronos, and it is
the only one still clearly worsening beyond 10 days. Chronos-2 and Financial
Chronos are flat across 10, 15 and 20 days.

Only one point sits below the naive benchmark: Chronos-2 at the corrected h=1
(0.158, i.e. 0.002 better than naive on 11 rounds). That is not enough to claim a
usable one-day edge.

## 2. How committed the forecasts are

Confidence is the largest of the five predicted probabilities. The L1 distance is
the total absolute departure from the uniform forecast [0.2, 0.2, 0.2, 0.2, 0.2].
They measure the same idea two ways, so they are shown side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
panels = [("mean_confidence", "mean confidence (largest probability)", 0.2, "uniform = 0.2"),
          ("mean_l1_from_uniform", "mean L1 distance from uniform", 0.0, "uniform = 0")]
for ax, (column, title, reference, note) in zip(axes, panels):
    for model, colour in COLOURS.items():
        block = diagnostics[diagnostics["model"] == model].sort_values("horizon_days")
        ax.plot(block["horizon_days"], block[column], marker="o",
                color=colour, label=SHORT[model])
    ax.axhline(reference, color="grey", linestyle="--", linewidth=1)
    ax.text(1.05, reference, note, color="grey", fontsize=8, va="bottom")
    ax.set_xscale("log")
    ax.set_xticks(HORIZONS)
    ax.set_xticklabels([str(h) for h in HORIZONS])
    ax.set_xlabel("forecast horizon (weekdays)")
    ax.set_title(title, fontsize=10)
axes[0].legend(fontsize=8)
fig.suptitle("Forecasts become more committed as the horizon lengthens", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "confidence_l1_vs_horizon.png", dpi=200)
plt.show()

Both panels rise monotonically for all three models, with h=15 sitting between its
neighbours everywhere. The ordering across models is also the ordering of their
RPS: Chronos-2 commits least and scores best, Chronos-T5 Base commits most and
scores worst.

Accuracy (in the table above) stays inside 0.222-0.267 against a chance level of
0.2 at every horizon, so the extra confidence is not matched by better quintile
identification. That mismatch is the experiment's central finding.

## 3. Distribution of per-asset confidence

The means above hide how much the individual assets differ. Outliers are omitted
so the boxes stay legible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for ax, (model, colour) in zip(axes, COLOURS.items()):
    block = clean[clean["model"] == model]
    data = [block.loc[block["horizon_days"] == h, "confidence"] for h in HORIZONS]
    parts = ax.boxplot(data, tick_labels=[str(h) for h in HORIZONS], showfliers=False,
                       widths=0.55, patch_artist=True, medianprops={"color": "black"})
    for patch in parts["boxes"]:
        patch.set_facecolor(colour)
        patch.set_alpha(0.45)
    ax.axhline(0.2, color="grey", linestyle="--", linewidth=1)
    ax.set_xlabel("forecast horizon (weekdays)")
    ax.set_title(SHORT[model], fontsize=10)
axes[0].set_ylabel("confidence (largest quintile probability)")
axes[0].text(0.6, 0.205, "uniform = 0.2", color="grey", fontsize=8)
fig.suptitle("Per-asset confidence: the whole distribution shifts up and widens",
             fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "confidence_distributions.png", dpi=200)
plt.show()

The median rises with horizon and the boxes get taller, so the models are not just
more confident on average - they increasingly disagree between assets about how
confident to be. Chronos-T5 Base is the extreme case: by 20 days its upper
quartile is far above where its median sat at one day.

## 4. How the five quintile probabilities change with horizon

**Why not simply average the five probabilities at each horizon?** Because that is
flat by construction. Each scenario ranks all 100 assets and assigns exactly 20 to
each quintile, so the cross-sectional mean of every quintile column is exactly 0.2
at every horizon - the cell below checks this. Such a plot would show five
identical bars and say nothing.

Instead each asset's five probabilities are **sorted from largest to smallest**
and then averaged across assets. This shows the average SHAPE of a forecast: how
much mass goes to the model's favourite quintile, its second favourite, and so on.
A flat line at 0.2 is an uninformative forecast; a steeper line is a more
concentrated one. The position labels refer to each asset's own ordering, so they
are not tied to any particular quintile.

In [ ]:
means = clean.groupby(["model", "horizon_days"])[RANK_COLUMNS].mean().round(6)
print("plain cross-sectional mean of each quintile column (flat by construction):")
print(means.head(5).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
positions = np.arange(1, 6)
shades = plt.cm.viridis(np.linspace(0.1, 0.85, len(HORIZONS)))

for ax, (model, colour) in zip(axes, COLOURS.items()):
    block = clean[clean["model"] == model]
    for h, shade in zip(HORIZONS, shades):
        vectors = block.loc[block["horizon_days"] == h, RANK_COLUMNS].to_numpy()
        profile = np.sort(vectors, axis=1)[:, ::-1].mean(axis=0)
        ax.plot(positions, profile, marker="o", color=shade, label=f"h={h}")
    ax.axhline(0.2, color="grey", linestyle="--", linewidth=1)
    ax.set_xticks(positions)
    ax.set_xticklabels(["1st", "2nd", "3rd", "4th", "5th"])
    ax.set_xlabel("quintile, ordered by each asset's own probability")
    ax.set_title(SHORT[model], fontsize=10)
axes[0].set_ylabel("mean probability")
axes[0].text(4.0, 0.215, "uniform = 0.2", color="grey", fontsize=8)
axes[2].legend(fontsize=8, title="horizon")
fig.suptitle("Average shape of the five-quintile forecast: more concentrated as the "
             "horizon grows", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "quintile_probability_profile.png", dpi=200)
plt.show()

The lines fan out with horizon for every model. Mass moves steadily from the
middle and bottom of each asset's own ordering into its top choice, and the
bottom quintile is progressively emptied - for Chronos-T5 Base the least-likely
quintile falls from about 0.055 at one day to about 0.020 at twenty, while its
favourite rises from about 0.40 to about 0.52.

So the answer to the question this figure was made for is: the distributions
become more CONCENTRATED with horizon, and the shape change is a smooth
redistribution rather than a change of form. Chronos-2 stays closest to the flat
uniform line at every horizon, which is consistent with it scoring best.

A caution on reading this: the sorting is done per asset, so the horizontal axis
is a within-asset ordering, not quintile 1 to 5. It shows how peaked the forecasts
are, not which quintiles are favoured, and a more peaked forecast is only better
if the peak is in the right place - which the accuracy column shows it generally
is not.

## What the horizon experiment shows

- **Confidence rises with horizon** for all three models, in both the mean and the
  full distribution, and the forecast shape becomes steadily more concentrated.
- **Accuracy does not follow.** It stays in 0.222-0.267 against a 0.2 chance level
  at every horizon, and the mean rank correlation between predicted and realised
  returns stays within +/-0.09 with no shared trend (both in the table above).
- **So RPS worsens with horizon.** Short-horizon forecasts score better mainly
  because they are closer to uninformative, not because they are more informative.
  A near-uniform forecast automatically scores close to the naive 0.16, which is
  better than any of these models achieves at 20 days.

A POSSIBLE mechanism, not established by this experiment: accumulating daily
forecast movements over longer horizons lets simulated paths separate more
strongly, which produces more concentrated cross-sectional quintile assignments
and therefore higher confidence. If that separation carries no genuine predictive
information, confidence rises without accuracy following. Nothing here tests that.

Caveats: h=1 excludes the degenerate Memorial Day round except where the hollow
marker is shown; the five horizons are nested slices of one set of 500
trajectories, so they are not independent replicates; and all of this rests on 12
rounds of 100 assets.